# CIFAR-10 Triplet Loss Training

**Before running:** Runtime → Change runtime type → T4 GPU  
Then: Runtime → Run all


In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi | head -12


In [ ]:
%%writefile model.py
"""Shared, swappable feature extractors for the CIFAR-10 retrieval pipeline.

Two extractors are provided so the retrieval pipeline can plug in either one
without any other changes:

* ``BaselineExtractor`` -- the original baseline (member 1): a pretrained
  ImageNet ResNet18 with the classifier removed, producing L2-normalized
  512-d features.
* ``EmbeddingNet`` -- the metric-learning model (member 2): the same ResNet18
  backbone followed by a projection head mapping 512 -> 128, trained with the
  batch-hard triplet loss in ``losses.py``.

``build_feature_extractor(mode, ...)`` returns ``(model, transform)`` and is the
single entry point used by ``retrieval.py`` (and reusable by member 3 for the
part c) evaluation). Both extractors return already L2-normalized embeddings,
so callers can rank by a plain dot product (cosine similarity).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

# ImageNet statistics used to normalize inputs for the fine-tuned model.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

EMBEDDING_DIM = 128


def get_device():
    """Pick the best available device: CUDA, then Apple MPS, then CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _resnet18_backbone(pretrained=True):
    """ResNet18 with the final classifier removed -> output shape (B, 512, 1, 1)."""
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    backbone = models.resnet18(weights=weights)
    return nn.Sequential(*list(backbone.children())[:-1])


class BaselineExtractor(nn.Module):
    """Pretrained ResNet18 backbone returning L2-normalized 512-d features.

    Reproduces the baseline behavior from the original ``retrieval.py``.
    """

    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = _resnet18_backbone(pretrained=pretrained)

    def forward(self, x):
        feat = self.backbone(x).flatten(1)  # (B, 512)
        return F.normalize(feat, p=2, dim=1)


class EmbeddingNet(nn.Module):
    """ResNet18 backbone + projection head (512 -> ``embedding_dim``).

    The projection head is a single linear layer followed by a BatchNorm
    (BNNeck-style, which is known to help retrieval). The forward pass returns
    L2-normalized embeddings so that Euclidean distance and cosine similarity
    are equivalent for both training (triplet loss) and retrieval.
    """

    def __init__(self, embedding_dim=EMBEDDING_DIM, pretrained=True):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.backbone = _resnet18_backbone(pretrained=pretrained)
        self.projection = nn.Sequential(
            nn.Linear(512, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x):
        feat = self.backbone(x).flatten(1)  # (B, 512)
        emb = self.projection(feat)          # (B, embedding_dim)
        return F.normalize(emb, p=2, dim=1)


def baseline_transform():
    """Preprocessing for the baseline extractor (matches the original code)."""
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])


def trained_transform():
    """Preprocessing for the fine-tuned model (adds ImageNet normalization).

    64x64 is enough for CIFAR-10 metric learning and ~10x faster to train than
    224x224 (CIFAR-10 images are natively 32x32, so 224 adds no information).
    """
    return transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def build_feature_extractor(mode="trained", ckpt_path=None, device=None):
    """Build a feature extractor and its matching eval-time transform.

    Args:
        mode: ``"baseline"`` for the pretrained ResNet18 (512-d) or
            ``"trained"`` for the triplet-trained ``EmbeddingNet`` (128-d).
        ckpt_path: checkpoint saved by ``train.py`` (required for ``"trained"``).
        device: target device; defaults to :func:`get_device`.

    Returns:
        ``(model, transform)`` -- model is in ``eval()`` mode on ``device``.
    """
    if device is None:
        device = get_device()

    if mode == "baseline":
        model = BaselineExtractor(pretrained=True)
        transform = baseline_transform()
    elif mode == "trained":
        if ckpt_path is None:
            raise ValueError("ckpt_path is required when mode='trained'")
        ckpt = torch.load(ckpt_path, map_location=device)
        embedding_dim = ckpt.get("embedding_dim", EMBEDDING_DIM)
        # Backbone weights are overwritten by the checkpoint, so skip the
        # (slow) pretrained download here.
        model = EmbeddingNet(embedding_dim=embedding_dim, pretrained=False)
        model.load_state_dict(ckpt["state_dict"])
        transform = trained_transform()
    else:
        raise ValueError(f"Unknown mode: {mode!r} (expected 'baseline' or 'trained')")

    model = model.to(device)
    model.eval()
    return model, transform


In [ ]:
%%writefile losses.py
"""Batch-hard triplet loss with in-batch hard negative mining.

Implements the "batch-hard" strategy from Hermans et al., *In Defense of the
Triplet Loss for Person Re-Identification* (https://arxiv.org/abs/1703.07737):

For every sample in the batch (used as an anchor) we mine, **within the same
batch**, the hardest positive (the same-class sample that is *farthest* away)
and the hardest negative (the different-class sample that is *closest*). The
loss pushes the hardest positive closer than the hardest negative by at least a
margin.

Because CIFAR-10 has only 10 classes, an ordinary shuffled batch of e.g. 128
images already contains many samples per class, so every anchor has valid
positives and negatives -- no dedicated P x K sampler is required.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class BatchHardTripletLoss(nn.Module):
    """Triplet margin loss with batch-hard mining.

    Args:
        margin: desired separation between the hardest positive and hardest
            negative distance.
    """

    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        """Compute the batch-hard triplet loss.

        Args:
            embeddings: (B, D) tensor, expected to be L2-normalized.
            labels: (B,) integer class labels.

        Returns:
            ``(loss, active_fraction)`` where ``active_fraction`` is the share
            of anchors whose triplet still violates the margin (a useful
            training signal: it should fall as the model improves).
        """
        # Pairwise Euclidean distances between all embeddings in the batch.
        dist = torch.cdist(embeddings, embeddings, p=2)  # (B, B)

        labels = labels.view(-1, 1)
        same = labels == labels.t()                       # (B, B) same-class mask
        eye = torch.eye(same.size(0), dtype=torch.bool, device=same.device)

        positive_mask = same & ~eye   # same class, excluding self
        negative_mask = ~same         # different class

        # Hardest positive: largest distance among same-class samples.
        # Invalid entries set to -inf so they never win the max.
        pos_dist = dist.masked_fill(~positive_mask, float("-inf"))
        hardest_pos, _ = pos_dist.max(dim=1)

        # Hardest negative: smallest distance among different-class samples.
        # Invalid entries set to +inf so they never win the min.
        neg_dist = dist.masked_fill(~negative_mask, float("inf"))
        hardest_neg, _ = neg_dist.min(dim=1)

        # Only keep anchors that actually have both a positive and a negative
        # in the batch (guards against degenerate batches).
        valid = torch.isfinite(hardest_pos) & torch.isfinite(hardest_neg)
        if valid.sum() == 0:
            return embeddings.new_zeros(()), embeddings.new_zeros(())

        hardest_pos = hardest_pos[valid]
        hardest_neg = hardest_neg[valid]

        losses = F.relu(hardest_pos - hardest_neg + self.margin)
        loss = losses.mean()
        active_fraction = (losses > 0).float().mean()
        return loss, active_fraction


In [ ]:
%%writefile train.py
"""Fine-tune an embedding model on CIFAR-10 with the batch-hard triplet loss.

This is part b) "metric learning": we start from an ImageNet-pretrained ResNet18
and learn a 128-d embedding space (via the projection head in ``model.py``)
using the batch-hard triplet loss in ``losses.py``. The resulting checkpoint is
consumed by ``retrieval.py`` simply by swapping the feature extractor.

Run:
    python src/train.py

The script self-bootstraps the CIFAR-10 download into ``data/``.
"""

import os
import ssl

import certifi
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm import tqdm

from losses import BatchHardTripletLoss
from model import EmbeddingNet, get_device, IMAGENET_MEAN, IMAGENET_STD

# Allow CIFAR-10 to download over HTTPS on macOS (mirrors dataset.py).
ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=certifi.where()
)

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
DATA_ROOT = "data"
CKPT_DIR = "checkpoints"
CKPT_PATH = os.path.join(CKPT_DIR, "triplet_resnet18.pt")

EMBEDDING_DIM = 128
BATCH_SIZE = 128
EPOCHS = 12
LR = 3e-4
WEIGHT_DECAY = 1e-4
MARGIN = 0.2
NUM_WORKERS = 4


def build_train_loader():
    """CIFAR-10 train split with light augmentation + ImageNet normalization.

    The transform must match ``model.trained_transform`` at inference time
    (same resize + normalization) so the embeddings are comparable.
    """
    train_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    dataset = CIFAR10(root=DATA_ROOT, train=True, download=True,
                      transform=train_transform)
    # With only 10 classes, a shuffled batch of 128 contains every class many
    # times over, so batch-hard mining always finds valid positives/negatives.
    # (A P x K sampler would be the alternative for datasets with many classes.)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, drop_last=True)


def train():
    device = get_device()
    print("Using device:", device)

    train_loader = build_train_loader()

    model = EmbeddingNet(embedding_dim=EMBEDDING_DIM, pretrained=True).to(device)
    criterion = BatchHardTripletLoss(margin=MARGIN)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_active = 0.0
        n_batches = 0

        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
        for images, labels in progress:
            images = images.to(device)
            labels = labels.to(device)

            embeddings = model(images)
            loss, active = criterion(embeddings, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_active += active.item()
            n_batches += 1
            progress.set_postfix(loss=loss.item(), active=active.item())

        scheduler.step()
        print(f"Epoch {epoch}: loss={running_loss / n_batches:.4f} "
              f"active_triplets={running_active / n_batches:.3f}")

    os.makedirs(CKPT_DIR, exist_ok=True)
    torch.save({"state_dict": model.state_dict(),
                "embedding_dim": EMBEDDING_DIM}, CKPT_PATH)
    print("Saved checkpoint to", CKPT_PATH)
    return model


# ============================================================================
# --- TEMP: self-check, remove/comment before handoff to member 3 (part c) ----
# Quick Recall@K / mAP on a small subset just to confirm the trained model
# beats the baseline. Member 3 owns the real metrics in metrics.py/evaluate.py,
# so this block is intentionally self-contained and disposable.
# ============================================================================
def _self_check():
    import torch.nn.functional as F
    from torch.utils.data import Subset
    from model import build_feature_extractor

    device = get_device()
    GALLERY_N, QUERY_N, KS = 5000, 1000, (1, 5, 10)

    def extract(model, transform):
        gallery = Subset(CIFAR10(DATA_ROOT, train=True, download=True,
                                 transform=transform), range(GALLERY_N))
        query = Subset(CIFAR10(DATA_ROOT, train=False, download=True,
                               transform=transform), range(QUERY_N))
        g_loader = DataLoader(gallery, batch_size=64, num_workers=NUM_WORKERS)
        q_loader = DataLoader(query, batch_size=64, num_workers=NUM_WORKERS)

        def run(loader):
            feats, labs = [], []
            with torch.no_grad():
                for imgs, lab in tqdm(loader, desc="extract"):
                    feats.append(model(imgs.to(device)).cpu())
                    labs.append(lab)
            return torch.cat(feats), torch.cat(labs)

        return run(g_loader), run(q_loader)

    def metrics(qf, ql, gf, gl):
        sims = qf @ gf.t()                       # cosine sim (features normalized)
        ranking = sims.argsort(dim=1, descending=True)
        rel = (gl[ranking] == ql.view(-1, 1)).float()  # (Q, G) relevance
        recall = {k: (rel[:, :k].sum(dim=1) > 0).float().mean().item() for k in KS}
        # mean average precision over the full ranking
        cum_hits = rel.cumsum(dim=1)
        ranks = torch.arange(1, rel.size(1) + 1).float()
        precision_at_hits = (cum_hits / ranks) * rel
        ap = precision_at_hits.sum(dim=1) / rel.sum(dim=1).clamp(min=1)
        return recall, ap.mean().item()

    for mode in ("baseline", "trained"):
        model, transform = build_feature_extractor(
            mode, ckpt_path=CKPT_PATH if mode == "trained" else None,
            device=device)
        (gf, gl), (qf, ql) = extract(model, transform)
        recall, mAP = metrics(qf, ql, gf, gl)
        recall_str = ", ".join(f"R@{k}={recall[k]:.3f}" for k in KS)
        print(f"[{mode}] {recall_str}, mAP={mAP:.3f}")
# --- END TEMP -----------------------------------------------------------------


if __name__ == "__main__":
    train()
    _self_check()  # TEMP: remove before handoff


In [ ]:
!python train.py


## Download checkpoint
Run the cell below to download `triplet_resnet18.pt` to your computer.
Put the file in `deep-learning-project-6-main/src/checkpoints/`.


In [ ]:
from google.colab import files
files.download('checkpoints/triplet_resnet18.pt')
